# F1 (YOLO26s-OBB) — Kaggle Runner

Entry point for the F1 family of the ablation study on **Dual Tesla T4**. Every step lives in `src/`; this notebook only orchestrates them so the logic stays testable and identical across sessions.

The notebook runs in two modes:

1. **Smoke run** (`--smoke-test`): the production recipe on 10 images per split for 3 epochs. It validates the whole path — dependency resolution, dual-GPU DDP, dataset linking, checkpoint saving and Google Drive upload — in a few minutes instead of hours.
2. **Production run**: cap of 40 epochs with `patience=5` on the full 43k-frame split.

Only the amount of work differs between them. The optimizer, learning rate, image size, augmentation profile, AMP setting and seed are identical, so a green smoke run is evidence about the real run and not about a separate code path.

### Conditions

| Condition | Training images | Augmentation |
|---|---|---|
| `c1` | Raw 640×360 | Minimal (geometric only) |
| `c2` | Raw 640×360 | Classic YOLO (mosaic, mixup, copy-paste) |
| `c3` | LaMa-cleaned 640×360 | Minimal, identical to `c1` |

`c1` and `c3` share every hyperparameter by construction: the only difference is the pixel content of the training images, which is what the intra-family gain measures.

### Requirements for this session

- Accelerator: **GPU T4 × 2**
- Internet: **on** (the repository is cloned and the package installed from PyPI)
- Dataset attached: `alvaroquispeunsa/mtc-challenge`
- Secret attached: `DRIVE_TOKEN_JSON` (Add-ons → Secrets)

Secrets cannot be attached through the Kaggle API, so this is the one step that has to be done in the notebook editor. Without the secret the smoke run still executes and still verifies the GPUs, but the preflight reports `drive` as failed and step 5 refuses to clear the environment. A production run stops immediately instead: a run that cannot persist its weights would spend the GPU quota for nothing.

## 0. Hardware inventory

Read the accelerator before installing anything. If only one GPU shows up here, the session was started with the wrong accelerator and there is no point in continuing.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv

import torch

print(f"\ntorch          : {torch.__version__}")
print(f"CUDA runtime   : {torch.version.cuda}")
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"Device count   : {torch.cuda.device_count()}")

if torch.cuda.device_count() < 2:
    raise RuntimeError(
        "This notebook targets 2 GPUs. Set the accelerator to 'GPU T4 x2' and restart."
    )

## 1. Clone the repository and install the package

A sparse clone brings only `experiments/`, and the package is installed in editable mode with the `[cloud]` extra. That extra deliberately omits `torch` and `torchvision`: the Kaggle image already ships CUDA-enabled builds, and reinstalling them would either waste minutes or replace them with a CPU-only wheel.

In [ ]:
import os
from pathlib import Path

REPO_NAME = 'ia_article'
REPO_URL = 'https://github.com/unsa-semester-2026-A/ia_article.git'
BRANCH_NAME = '13-f1-kaggle-runner'

BASE_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
%cd {BASE_DIR}

REPO_PATH = Path(BASE_DIR) / REPO_NAME

if not REPO_PATH.exists():
    print(f"Cloning {REPO_NAME} (branch {BRANCH_NAME})...")
    !git clone -q --depth 1 --branch {BRANCH_NAME} --filter=blob:none --sparse {REPO_URL}
    %cd {REPO_NAME}
    !git sparse-checkout set experiments
    %cd experiments
else:
    print(f"Updating existing {REPO_NAME} repository...")
    %cd {REPO_NAME}
    !git fetch -q origin {BRANCH_NAME}
    !git checkout -q {BRANCH_NAME}
    !git reset -q --hard origin/{BRANCH_NAME}
    %cd experiments

current_dir = Path(os.getcwd())
if current_dir.name != 'experiments':
    raise RuntimeError(f"Directory navigation failed. Current path: {current_dir}")

!git log -1 --oneline

print("\nInstalling package in editable mode with [cloud] dependencies...")
%pip install -q -e .[cloud]

## 2. Unit tests

The co-located tests cover the trainer, the GPU sampler, the Drive I/O layer and the preflight checks. Running them here catches a dependency that resolved to an incompatible version before any GPU time is spent.

In [ ]:
!python -m pytest src/ -q --no-header -p no:cacheprovider

## 3. Preflight verification

Six checks, each answering a question that would otherwise only surface hours into a run:

| Check | Question |
|---|---|
| `dependencies` | Does every required package import, and at which version? |
| `gpus` | Are both T4s visible to torch and to the driver? |
| `labels` | Do both label splits exist and how many frames does each hold? |
| `image_sets` | Do the raw and LaMa sets have the same count and the same resolution? |
| `drive` | Do the credentials resolve and are both destination folders reachable? |
| `nccl_all_reduce` | Can two processes complete a collective over NCCL? |

The `image_sets` check is the one that protects the science: C1 and C3 must differ only in pixel content, so a different image count or resolution would make the ablation attribute to LaMa an effect caused by the data. The `nccl_all_reduce` check matters because Ultralytics multi-GPU training is DDP over NCCL; if the collective cannot complete, the run either hangs or silently degrades to a single GPU.

In [ ]:
DATASET_DIR = '/kaggle/input/mtc-challenge'
PREFLIGHT_REPORT = '/kaggle/working/preflight_report.json'

!python -m src.training.preflight \
    --labels-dir {DATASET_DIR}/yolo_obb_labels \
    --raw-images-dir {DATASET_DIR}/train_resized/train \
    --lama-images-dir {DATASET_DIR}/smart_lama_corrected/train \
    --expected-gpus 2 \
    --output {PREFLIGHT_REPORT}

## 4. Smoke run

Ten images per split, three epochs, everything else as in production. It writes to its own run name (`f1_c1_smoke`), so its artifacts never collide with real results in Drive, and it always starts cold instead of resuming.

Watch for three things in the output:

- `DDP` in the Ultralytics banner and a command line with `--nproc_per_node 2`.
- The `GPU USAGE` block at the end reporting a non-trivial memory peak on **both** devices.
- A Drive upload line per artifact, with names prefixed `f1_c1_smoke_`.

In [ ]:
!python -m src.training.trainers.train_base_1 \
    --condition c1 \
    --smoke-test \
    --smoke-images 10 \
    --smoke-epochs 3

## 5. Verdict

Reads the artifacts written by the previous two steps and decides whether the environment is cleared for a production run. This is deliberately a separate cell: it fails loudly rather than leaving a warning buried in hundreds of lines of training log.

In [ ]:
import json
from pathlib import Path

problems = []

report_path = Path('/kaggle/working/preflight_report.json')
if report_path.exists():
    report = json.loads(report_path.read_text())
    print("PREFLIGHT")
    for check in report['checks']:
        print(f"  {'PASS' if check['passed'] else 'FAIL'}  {check['name']}")
        if not check['passed']:
            problems.append(f"preflight check '{check['name']}' failed")
else:
    problems.append('preflight report missing: step 3 did not complete')

metrics_paths = sorted(Path('/kaggle/working/runs').glob('f1_*_smoke/training_metrics.json'))
if metrics_paths:
    metrics = json.loads(metrics_paths[-1].read_text())
    hardware = metrics.get('hardware', {})
    sampling = hardware.get('gpu_sampling', {})
    print("\nSMOKE RUN")
    print(f"  epochs         : {metrics.get('total_epochs_completed')}")
    print(f"  elapsed        : {hardware.get('total_time_seconds')} s")
    print(f"  peak CPU RAM   : {hardware.get('peak_cpu_ram_gb')} GB")
    for device in sampling.get('devices', []):
        print(
            f"  GPU {device['index']} {device['name']}: "
            f"peak {device['peak_memory_used_mib']:.0f} MiB, "
            f"mean util {device['mean_utilization_pct']:.0f}%"
        )
    if not hardware.get('multi_gpu_verified'):
        problems.append('training did not engage every expected GPU')
else:
    problems.append('smoke run metrics missing: step 4 did not complete')

print("\n" + "=" * 60)
if problems:
    print('NOT CLEARED FOR PRODUCTION')
    for problem in problems:
        print(f'  - {problem}')
else:
    print('CLEARED FOR PRODUCTION')
print("=" * 60)

## 6. Production run

Only run this once step 5 reports `CLEARED FOR PRODUCTION`.

Cap of 40 epochs with `patience=5`. The pilot run reached its best mAP50-95 around epoch 6 and the following 33 epochs added roughly 0.01, so the expected stop is between epochs 11 and 16, or about 3 hours at ~14.7 min per epoch.

Change `--condition` to launch `c2` or `c3`. If a Kaggle session times out, re-running this cell resumes from this run's `last.pt` in the checkpoints folder.

In [ ]:
# !python -m src.training.trainers.train_base_1 --condition c1